# Tcf4 RNA-velocity feature map

Remote source reviewed from `20260103-p38/TCF4-dy-feature.ipynb`. Run from top to bottom after configuring `config/paths.*`.


In [ ]:
# Centralized paths, deterministic seed, and scheduler-aware thread limits.
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "config" / "paths.py").exists() else Path.cwd().parent
if not (repo_root / "config" / "paths.py").exists():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory.")
sys.path.insert(0, str(repo_root))
from config.paths import P38_THREADS, RANDOM_SEED, legacy_path, output_path, p38_path


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.sparse import issparse
import os
from IPython.display import IFrame, display

# ==============================================================================
# 1. 确保数据已加载
# ==============================================================================
if 'adata_wt' not in locals():
    print("Loading WT data...")
    path_wt = p38_path("20260103-p38/final_results/adata_wt_dynamo_quantified.h5ad")
    adata_wt = dyn.read_h5ad(path_wt)

if 'adata_ko' not in locals():
    print("Loading KO data...")
    path_ko = p38_path("20260103-p38/final_results/adata_ko_dynamo_quantified.h5ad")
    adata_ko = dyn.read_h5ad(path_ko)

# ==============================================================================
# 2. 最终修复版绘图函数 (无报错 + 蓬松效果)
# ==============================================================================
def plot_tcf4_visual_final(save_name="gene_velocity_Tcf4_final"):
    # 确定基因名
    target_gene = "Tcf4"
    if target_gene not in adata_wt.var_names: 
        target_gene = "TCF4"
    
    print(f"\nProcessing: {target_gene} (Style: Fluffy/Connected)...")

    # 创建画布
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    
    # --- 1. 确定 Layer (优先使用平滑层 velocity_S) ---
    if 'velocity_S' in adata_wt.layers.keys():
        layer_key = 'velocity_S'
    else:
        print("Warning: 'velocity_S' not found, using raw 'velocity'.")
        layer_key = 'velocity'
    
    # --- 2. 计算统一的对称标尺 ---
    d_wt = adata_wt[:, target_gene].layers[layer_key]
    if issparse(d_wt): d_wt = d_wt.toarray()
    d_ko = adata_ko[:, target_gene].layers[layer_key]
    if issparse(d_ko): d_ko = d_ko.toarray()
    
    vals = np.concatenate([d_wt.flatten(), d_ko.flatten()])
    vmin_raw, vmax_raw = np.percentile(vals, 2), np.percentile(vals, 98)
    
    # 强制对称标尺
    limit = max(abs(vmin_raw), abs(vmax_raw))
    if limit < 0.1: limit = 0.1 
    vmin, vmax = -limit, limit

    # --- 3. 绘图 WT (pointsize=0.1, alpha=0.6) ---
    dyn.pl.scatters(adata_wt, basis='umap', color=target_gene, layer=layer_key,
                    ax=axes[0], show_legend='off', 
                    pointsize=0.1, alpha=0.6,  # <--- 保持这个参数以获得蓬松感
                    vmin=vmin, vmax=vmax, save_show_or_return='return')
    axes[0].set_title(f"WT: {target_gene}", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("UMAP1")
    axes[0].set_ylabel("UMAP2")

    # --- 4. 绘图 KO (pointsize=0.1, alpha=0.6) ---
    dyn.pl.scatters(adata_ko, basis='umap', color=target_gene, layer=layer_key,
                    ax=axes[1], show_legend='off', 
                    pointsize=0.1, alpha=0.6,  # <--- 保持这个参数以获得蓬松感
                    vmin=vmin, vmax=vmax, save_show_or_return='return')
    axes[1].set_title(f"KO: {target_gene}", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("UMAP1")
    axes[1].set_ylabel("UMAP2")

    # 标题
    fig.suptitle(f"Key Driver Analysis: {target_gene}", fontsize=14, fontweight='bold', y=0.98)
    
    # --- 关键修改：手动调整布局，替代 bbox_inches='tight' ---
    # rect 参数留出顶部给 suptitle
    fig.tight_layout(rect=[0, 0, 1, 0.95]) 

    # --- 5. 保存 ---
    save_dir = p38_path("20260103-p38/final_results/plots")
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/{save_name}.pdf"
    
    # ！！！这里去掉了 bbox_inches='tight' ！！！
    fig.savefig(save_path, format='pdf') 
    
    print(f"Successfully saved PDF to: {save_path}")
    
    # 关闭 Matplotlib 窗口
    plt.close(fig) 
    
    # 展示 PDF
    print("Displaying generated PDF below:")
    display(IFrame(save_path, width=900, height=500))

# ==============================================================================
# 执行
# ==============================================================================
plot_tcf4_visual_final()

/home/gonglihao/miniconda3/envs/dynamo_env/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/gonglihao/miniconda3/envs/dynamo_env/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/gonglihao/miniconda3/envs/dynamo_env/lib/python3.8

Loading WT data...
Loading KO data...

Processing: Tcf4 (Style: Fluffy/Connected)...
|-----------> plotting with basis key=X_umap
|-----------> plotting with basis key=X_umap


/home/gonglihao/my-vscode-tmp/ipykernel_145509/54957519.py:79: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.95])


Successfully saved PDF to: /data3/Group8/gonglihao/项目/p38/20260103-p38/final_results/plots/gene_velocity_Tcf4_final.pdf
Displaying generated PDF below:
